# Step 07 — Chain of Thought

🇬🇧 **English** (this notebook) · 🇩🇪 Deutsch — not yet available.

Same component structure as Step 05, with one addition: an explicit instruction asking the model to reason through the problem before giving its final answer.

## Learning objective

By the end of this notebook, you will:

- Understand what "chain-of-thought" (CoT) prompting means, and the difference between zero-shot CoT and few-shot CoT
- Have added a `reasoning` component to a system-message template and observed its effect
- Be able to judge whether visible reasoning changes the model's *conclusion*, or just adds more text on the way to the same one

## Prerequisites

- [Step 05 — Prompt Template](step_05_prompt_template.ipynb) completed — this notebook reuses its component structure directly
- The same `.env` setup as the previous steps

## Background

Chain-of-thought prompting asks the model to reason step by step *before* stating its final answer, instead of jumping straight to a conclusion. Two related findings established this:

> Wei, J., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. NeurIPS 2022. [arXiv:2201.11903](https://arxiv.org/abs/2201.11903) — showed that including full worked reasoning chains as few-shot examples improves performance on multi-step problems.

> Kojima, T., et al. (2022). *Large Language Models are Zero-Shot Reasoners*. NeurIPS 2022. [arXiv:2205.11916](https://arxiv.org/abs/2205.11916) — showed a simpler version gets most of the same benefit: no worked examples needed, just a short instruction like "think through this step by step." This notebook uses that zero-shot version.

## How this works

Run the Setup cell once, then the exercise cell:

1. **Setup** loads your API keys and creates the `llm` object, same as before.
2. **The exercise cell** reuses Step 05's exact six components (`persona`, `instruction`, `context`, `data_format`, `audience`, `tone`) and adds one more: `reasoning`, a single line telling the model to think step by step before answering.
3. Everything else — how the components are joined into `system_message`, how the call is made — is identical to Step 05. `reasoning` is the only new variable.

In [ ]:
import os

# CrewAI's telemetry tries to reach its backend over the network on import;
# on a restricted/firewalled connection this can hang for a long time with no
# error. Disable it before crewai is imported.
os.environ.setdefault('CREWAI_DISABLE_TELEMETRY', 'true')

from dotenv import load_dotenv
from crewai import LLM
from IPython.display import Markdown, display

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

In [ ]:
# ── System message — same components as 2b, plus an explicit reasoning
# instruction telling the model to think step by step before answering ────────
persona       = "You are a helpful assistant.\n"
instruction   = "You are an expert in EU AI Act compliance.\n"
context       = "You are assisting a B2B SaaS company that uses LLMs in its product.\n"
data_format   = "Provide your response in bullet points.\n"
audience      = "The output will be read by legal professionals and compliance officers.\n"
tone          = "Be professional and concise.\n"
reasoning     = "First, think through the problem step by step. Then, provide your final answer.\n"

system_message = persona + instruction + context + data_format + audience + tone + reasoning

# ── User message — the actual question ───────────────────────────────────────
text = "EU AI Act compliance requirements for a B2B SaaS company that uses LLMs in its product"
user_message = f"Topic: {text}\n"

output = llm.call(
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
)
display(Markdown(output))

## Your task

1. Run the cells (Setup first, then the exercise cell).

2. Compare this output to Step 05's — same persona/instruction/context/audience/tone, plus the `reasoning` line. Does the reasoning instruction produce noticeably different conclusions, or just more text on the way to the same bullet points?

3. Remove just the `reasoning` line (set it to `""`) and re-run. Is the *answer* different, or only the visible thinking?

4. Note what you observed — it's evidence for `REPORT.md`'s Section 4.3 (Prompting Techniques: Chain of Thought).

## Shortcomings

Visible reasoning is not the same as *correct* reasoning — a model can produce a fluent, plausible-looking chain of thought that leads to the wrong conclusion, or state a conclusion that its own preceding reasoning doesn't actually support (task 2 above asks you to check whether that happened here). Chain-of-thought also only explores **one** line of reasoning; if that line starts off on the wrong track, nothing recovers it.

[Step 08](step_08_tree_of_thought.ipynb) addresses exactly that: exploring several reasoning paths in parallel instead of committing to just one.

## Resources for further reading

- Wei, J., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. NeurIPS 2022. [arXiv:2201.11903](https://arxiv.org/abs/2201.11903)
- Kojima, T., et al. (2022). *Large Language Models are Zero-Shot Reasoners*. NeurIPS 2022. [arXiv:2205.11916](https://arxiv.org/abs/2205.11916)